# 01 - Analyse exploratoire du dataset

Notebook d'EDA sur le dataset consolide `01_data/processed/dataset.parquet`.

Pre-requis : avoir lance les scripts d'acquisition + `build_dataset.py`.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 90

ROOT = Path('..').resolve()
print('Racine projet :', ROOT)

## 1. Chargement du dataset

In [ ]:
df = pd.read_parquet(ROOT / '01_data/processed/dataset.parquet')
print(f'Dataset : {len(df):,} lignes, {df.shape[1]} colonnes')
df.head()

In [ ]:
df.info()

## 2. Repartition globale

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df['label'].map({0: 'reelle', 1: 'synthetique'}).value_counts().plot.bar(
    ax=axes[0], color=['#2a9d8f', '#e76f51'], rot=0)
axes[0].set_title('Distribution des classes')
axes[0].set_ylabel('Nombre d\'images')

df['source'].value_counts().plot.bar(ax=axes[1], color='steelblue', rot=45)
axes[1].set_title('Distribution par source')

df['split'].value_counts().plot.bar(ax=axes[2], color='darkslateblue', rot=45)
axes[2].set_title('Distribution par split')

plt.tight_layout()
plt.savefig(ROOT / '06_reports/figures/eda_distribution_globale.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Verification de l'equilibre par split et par classe
pivot = pd.crosstab([df['split'], df['source']], df['label'].map({0: 'reel', 1: 'fake'}))
pivot

## 3. Categories habitation

In [ ]:
domain = df[df['category'].isin(['water', 'fire', 'glass', 'vandalism'])].copy()
if len(domain) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    pivot = domain.groupby(['category', 'label']).size().unstack(fill_value=0)
    pivot.columns = ['reelles', 'synthetiques']
    pivot.plot.bar(ax=ax, color=['#2a9d8f', '#e76f51'], rot=0)
    ax.set_title('Repartition par categorie de degat (domaine habitation)')
    ax.set_ylabel('Nombre d\'images')
    plt.tight_layout()
    plt.savefig(ROOT / '06_reports/figures/eda_categories_habitation.png', dpi=120, bbox_inches='tight')
    plt.show()
else:
    print('Aucune donnee dans le domaine habitation pour le moment')

## 4. Metadonnees EXIF

On verifie une intuition : les images synthetiques ont tendance a ne pas avoir d'EXIF. Si c'est trop marque, le modele pourrait apprendre a tricher uniquement sur cette feature.

In [ ]:
exif_stats = df.groupby(df['label'].map({0: 'reelles', 1: 'synthetiques'}))[['exif_present', 'has_gps']].mean() * 100
exif_stats.columns = ['% avec EXIF', '% avec GPS']
exif_stats

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
exif_stats.plot.bar(ax=ax, rot=0, color=['#264653', '#f4a261'])
ax.set_title('Presence des metadonnees EXIF par classe')
ax.set_ylabel('%')
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig(ROOT / '06_reports/figures/eda_exif_par_classe.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Dimensions des images

In [ ]:
df['megapixels'] = df['width'] * df['height'] / 1e6
fig, ax = plt.subplots(figsize=(10, 4))
for lab, color in [(0, '#2a9d8f'), (1, '#e76f51')]:
    sub = df[df['label'] == lab]['megapixels'].dropna()
    if len(sub) > 0:
        ax.hist(sub, bins=40, alpha=0.6, color=color,
                label='reelles' if lab == 0 else 'synthetiques')
ax.set_xlabel('Megapixels')
ax.set_ylabel('Frequence')
ax.set_title('Distribution de la taille des images')
ax.legend()
plt.tight_layout()
plt.savefig(ROOT / '06_reports/figures/eda_dimensions.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Doublons (perceptual hash)

In [ ]:
if 'phash' in df.columns and df['phash'].notna().any():
    n_unique = df['phash'].nunique()
    print(f'Hashes uniques : {n_unique:,} / {len(df):,}')
    duplicates = df.groupby('phash').size().sort_values(ascending=False)
    duplicates = duplicates[duplicates > 1]
    print(f'Groupes de doublons : {len(duplicates)}')
    if len(duplicates) > 0:
        print(duplicates.head(10))
else:
    print('Aucun phash calcule (option --no-phash ?)')

## 7. Visualisation echantillon

In [ ]:
def show_grid(sub_df, title, n=8):
    sub_df = sub_df.sample(min(n, len(sub_df)), random_state=42)
    cols = min(4, len(sub_df))
    rows = (len(sub_df) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = np.array(axes).reshape(-1)
    for ax, (_, row) in zip(axes, sub_df.iterrows()):
        try:
            img = Image.open(ROOT / row['image_path'])
            ax.imshow(img)
        except Exception as e:
            ax.text(0.5, 0.5, f'erreur\n{e}', ha='center', va='center')
        ax.set_title(f"{row['source']} | {row['category']}", fontsize=9)
        ax.axis('off')
    for ax in axes[len(sub_df):]:
        ax.axis('off')
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

if (df['label'] == 0).any():
    show_grid(df[df['label'] == 0], 'Echantillon images REELLES')
if (df['label'] == 1).any():
    show_grid(df[df['label'] == 1], 'Echantillon images SYNTHETIQUES')

## 8. Synthese

A retenir pour le rapport :
- Volumetrie totale et repartition par classe
- Stratification correcte des splits
- Eventuel desequilibre par categorie habitation
- Risque de fuite par les EXIF si trop marque
- Diversite visuelle de l'echantillon